# 5.3 Local search for CSPs and problem structure: practice notebook

Repair a complete assignment instead of building one: min-conflicts solves huge N queens problems in a few steps. Then use the shape of the constraint graph: trees are solved in linear time, and cutsets turn near trees into trees.

How to use it: run the cells from top to bottom (Shift + Enter). First read and run the examples. Then try each task in its starter cell and run the check cell under it. The full solutions are at the end, so try on your own first.

Everything here is plain Python 3. No extra packages are needed.

## Setup

In [ ]:
# A small helper. check(tests) runs each test of a task and tells you which ones pass.
# Each test is a pair: the call to make, and the answer we expect.
import json

def check(tests):
    passed = 0
    for call, expected in tests:
        try:
            got = eval(call)
        except Exception as error:
            got = "error: " + str(error)
        same = json.dumps(got, sort_keys=True, default=repr) == json.dumps(expected, sort_keys=True, default=repr)
        if same:
            passed = passed + 1
            print("PASS ", call)
        else:
            print("FAIL ", call, " expected", expected, " but got", got)
    if passed == len(tests):
        print("All", len(tests), "tests pass. Well done!")
    else:
        print(passed, "of", len(tests), "tests pass. Keep going, or look at the solution below.")

## Examples from the chapter

### Min-conflicts on 4 queens

In [ ]:
# Min-conflicts for 4 queens. board[c] is the row of the queen in column c.
import random
random.seed(2)
n = 4
board = [random.randrange(n) for c in range(n)]   # a random complete assignment

def conflicts(col, row):
    count = 0
    for c in range(n):
        if c != col and (board[c] == row or abs(board[c] - row) == abs(c - col)):
            count = count + 1
    return count

print("start:", board)
for step in range(10):
    attacked = [c for c in range(n) if conflicts(c, board[c]) > 0]
    if len(attacked) == 0:
        print("solved:", board)
        break
    col = random.choice(attacked)                 # a variable in conflict, picked at random
    counts = [conflicts(col, r) for r in range(n)]
    best = min(counts)
    board[col] = random.choice([r for r in range(n) if counts[r] == best])
    print("column", col, "attacks per row", counts, "-> move to row", board[col])

### Solving a tree shaped CSP

In [ ]:
# Solving a tree shaped CSP with one pass backward and one pass forward.
# The tree: A is the root, B and C are children of A, D is a child of B. Every edge says "different colours".
order = ["A", "B", "C", "D"]                       # parents always come before children
parent = {"B": "A", "C": "A", "D": "B"}
domains = {"A": ["red", "green"], "B": ["red"], "C": ["red", "green"], "D": ["green", "red"]}

# backward pass: make each arc (parent, child) consistent, from the last child to the first
for child in reversed(order[1:]):
    p = parent[child]
    for v in domains[p][:]:
        partner = False
        for w in domains[child]:
            if w != v:
                partner = True
        if not partner:
            domains[p].remove(v)
            print("remove", v, "from", p, "because", child, "would have no colour left")
print("after the backward pass:", domains)

# forward pass: give each variable any value that fits its parent. No backtracking is ever needed.
value = {}
for x in order:
    for v in domains[x]:
        if x not in parent or v != value[parent[x]]:
            value[x] = v
            break
print("solution:", value)

## Your turn

### Task 1: Min-conflicts for N queens

Write `min_conflicts(board, max_steps, seed)`. board[c] is the row of the queen in column c. Call random.seed(seed) first. Each step: list the attacked columns. If there are none, return the board. Otherwise pick one with random.choice, count its conflicts in every row, and move it to a row with the fewest conflicts (random.choice among ties). After max_steps return the board as it is. A helper count_attacks is given for the checks.

Hint: Write a helper conflicts(col, row) inside the function. Build the list of attacked columns and the list of best rows with plain loops.

In [ ]:
import random

def count_attacks(board):
    count = 0
    for c1 in range(len(board)):
        for c2 in range(c1 + 1, len(board)):
            if board[c1] == board[c2] or abs(board[c1] - board[c2]) == c2 - c1:
                count = count + 1
    return count

def min_conflicts(board, max_steps, seed):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([('count_attacks(min_conflicts([0, 0, 0, 0, 0, 0, 0, 0], 300, 1))', 0), ('count_attacks(min_conflicts([4, 5, 6, 3, 4, 5, 6, 5], 300, 3))', 0), ('count_attacks(min_conflicts([0] * 20, 300, 2))', 0), ('min_conflicts([1, 3, 0, 2], 10, 5)', [1, 3, 0, 2])])

### Task 2: Solve a tree CSP

Write `solve_tree(order, parent, domains)` for a tree CSP where every edge means "different values". order lists the variables with parents first. Do the backward pass (remove parent values with no different value left in the child), then the forward pass (first value in the domain that differs from the parent). Return the assignment dict, or None if a domain becomes empty.

Hint: Walk the order backwards for the first pass and forwards for the second. Copy the domains first.

In [ ]:
def solve_tree(order, parent, domains):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("solve_tree(['A', 'B', 'C', 'D'], {'B': 'A', 'C': 'A', 'D': 'B'}, {'A': ['red', 'green'], 'B': ['red'], 'C': ['red', 'green'], 'D': ['green', 'red']})", {'A': 'green', 'B': 'red', 'C': 'red', 'D': 'green'}), ("solve_tree(['A', 'B'], {'B': 'A'}, {'A': ['x'], 'B': ['x']})", None), ("solve_tree(['A'], {}, {'A': ['x', 'y']})", {'A': 'x'})])

## Solutions

Try the tasks yourself first. Running a solution cell replaces your own function with the model one.

### Solution to task 1: Min-conflicts for N queens

With random choices the search almost never cycles. Try the same board with a different seed: the solution may differ, but it is found in a few dozen steps. If you always picked the first attacked column and the first best row instead, the search would often go round in circles.

In [ ]:
import random

def count_attacks(board):
    count = 0
    for c1 in range(len(board)):
        for c2 in range(c1 + 1, len(board)):
            if board[c1] == board[c2] or abs(board[c1] - board[c2]) == c2 - c1:
                count = count + 1
    return count

def min_conflicts(board, max_steps, seed):
    random.seed(seed)
    board = board[:]
    n = len(board)

    def conflicts(col, row):
        count = 0
        for c in range(n):
            if c != col and (board[c] == row or abs(board[c] - row) == abs(c - col)):
                count = count + 1
        return count

    for step in range(max_steps):
        attacked = []
        for c in range(n):
            if conflicts(c, board[c]) > 0:
                attacked.append(c)
        if len(attacked) == 0:
            return board
        col = random.choice(attacked)
        counts = []
        for r in range(n):
            counts.append(conflicts(col, r))
        best = min(counts)
        rows = []
        for r in range(n):
            if counts[r] == best:
                rows.append(r)
        board[col] = random.choice(rows)
    return board

In [ ]:
check([('count_attacks(min_conflicts([0, 0, 0, 0, 0, 0, 0, 0], 300, 1))', 0), ('count_attacks(min_conflicts([4, 5, 6, 3, 4, 5, 6, 5], 300, 3))', 0), ('count_attacks(min_conflicts([0] * 20, 300, 2))', 0), ('min_conflicts([1, 3, 0, 2], 10, 5)', [1, 3, 0, 2])])

### Solution to task 2: Solve a tree CSP

B can only be red, so A loses red in the backward pass, exactly as in the lesson.

In [ ]:
def solve_tree(order, parent, domains):
    dom = {}
    for v in domains:
        dom[v] = list(domains[v])
    for child in reversed(order[1:]):
        p = parent[child]
        for v in dom[p][:]:
            partner = False
            for w in dom[child]:
                if w != v:
                    partner = True
            if not partner:
                dom[p].remove(v)
        if len(dom[p]) == 0:
            return None
    value = {}
    for x in order:
        chosen = None
        for v in dom[x]:
            if x not in parent or v != value[parent[x]]:
                chosen = v
                break
        if chosen is None:
            return None
        value[x] = chosen
    return value

In [ ]:
check([("solve_tree(['A', 'B', 'C', 'D'], {'B': 'A', 'C': 'A', 'D': 'B'}, {'A': ['red', 'green'], 'B': ['red'], 'C': ['red', 'green'], 'D': ['green', 'red']})", {'A': 'green', 'B': 'red', 'C': 'red', 'D': 'green'}), ("solve_tree(['A', 'B'], {'B': 'A'}, {'A': ['x'], 'B': ['x']})", None), ("solve_tree(['A'], {}, {'A': ['x', 'y']})", {'A': 'x'})])